# 02 — Dixon-Coles model

Fit the model, read off team ratings, sanity-check against FIFA rankings, and make an example prediction.

In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.dixon_coles import DixonColesModel

df = pd.read_parquet("../data/processed/matches.parquet")
# Time decay down-weights old games, so we fit on the modern era for speed
# (a full fit takes ~40s). xi was tuned in notebook 03.
df_fit = df[df["date"] >= "2010-01-01"]
model = DixonColesModel().fit(df_fit, xi=0.0019)
print("home advantage (gamma):", round(model.home_adv, 3), "| rho:", round(model.rho, 4))

home advantage (gamma): 0.284 | rho: -0.1389


In [2]:
ratings = pd.DataFrame({"team": model.teams, "attack": model.attack, "defense": model.defense})
ratings["net"] = ratings["attack"] - ratings["defense"]   # higher = stronger overall
ratings.sort_values("net", ascending=False).head(15)

,team,attack,defense,net
37,Brazil,1.218199,-1.160837,2.379036
12,Argentina,1.151618,-1.093485,2.245103
58,Colombia,0.953804,-1.062523,2.016327
258,Spain,1.023230,-0.850418,1.873648
293,Uruguay,0.905174,-0.842447,1.747621
79,Ecuador,0.848016,-0.776114,1.624130
95,France,0.837097,-0.774125,1.611222
84,England,0.748007,-0.837057,1.585064
213,Portugal,0.885959,-0.693781,1.579740
127,Iran,0.712733,-0.820171,1.532904


In [3]:
from src.data import load_rankings, latest_rankings
ranks = latest_rankings(load_rankings("../data/raw/fifa_ranking.csv"))
ratings["fifa_rank"] = ratings["team"].map(ranks)
overlap = ratings.dropna(subset=["fifa_rank"])
corr = overlap[["net", "fifa_rank"]].corr().iloc[0, 1]
print(f"matched {len(overlap)} teams to FIFA ranks")
print("corr(net strength, FIFA rank):", round(corr, 3), "(expect strongly negative)")

matched 191 teams to FIFA ranks
corr(net strength, FIFA rank): -0.916 (expect strongly negative)


In [4]:
print("Brazil vs Croatia (neutral):", {k: round(v, 3) for k, v in
      model.predict_result("Brazil", "Croatia", neutral=True).items()})
mat = model.score_matrix("Brazil", "Croatia")
hg, ag = divmod(int(mat.argmax()), mat.shape[1])
print(f"most likely scoreline: Brazil {hg} - {ag} Croatia ({mat.max()*100:.1f}%)")

Brazil vs Croatia (neutral): {'home_win': 0.698, 'draw': 0.218, 'away_win': 0.084}
most likely scoreline: Brazil 2 - 0 Croatia (15.3%)
